# Warum deutsche Lehrer individuelle Förderung nicht leisten können.
Wie hat sich das Verhältnis zwischen wachsender Heterogenität in deutschen Schulklassen und verfügbarer Lehrerkapazität in den letzten Jahren entwickelt – und was zeigen die Leistungsergebnisse?

## Storyline

1. **Einstieg:** Zitat von Angela Merkel zur Bildungsrepublik Deutschland.
2. **Erste Beobachtung:** Die Zahl der Schüler je Lehrkraft sinkt seit Jahren – auf den ersten Blick eine positive Entwicklung.
3. **Relativierung:** Diese Kennzahl ist jedoch verzerrt, u. a. durch den steigenden Anteil an Teilzeitkräften sowie krankheitsbedingte Ausfälle.
4. **Veranschaulichung:** Anhand ausgewählter Einflussfaktoren (z. B. Teilzeitquote, Krankheitstage) zeigen wir, wie stark diese Effekte das tatsächliche Verhältnis verzerren.
5. **Zusätzliche Belastung:** Gleichzeitig wächst der Anteil an Kindern mit besonderem Förderbedarf, für deren Unterricht viele Lehrkräfte nicht ausreichend ausgebildet sind.
6. **Quantifizierung:** Daraus ergibt sich ein aktueller Lehrkräftemangel von ..., der sich laut Prognose bis 2035 auf ... verschärfen wird.
7. **Wahrnehmung der Lehrkräfte:** Umfragen zeigen, wie Lehrkräfte ihre Arbeitssituation aktuell empfinden – nicht zwingend kausal auf den Mangel zurückzuführen, aber ein wichtiges Stimmungsbild.
8. **Auswirkungen auf die Schüler:** Die PISA-Studie belegt einen erheblichen Leistungsrückgang deutscher Schülerinnen und Schüler.
9. **Interpretation:** Einordnung der Ergebnisse und Diskussion möglicher Zusammenhänge zwischen Lehrkräftemangel, Heterogenität und Leistungsentwicklung.

## Anzahl Schüler vs. Leher

In [ ]:
# Imports und Datenpfade
# Excel-Tabellen unter ./Statistiken/
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('Statistiken')
F_LEHRER = DATA / 'Lehrer an allgemeinbildenden Schulen.xlsx'
F_SCHUELER = DATA / 'Schüler an allgemeinbildenden Schulen.xlsx'

In [ ]:
def load(path):
    """Liest eine Excel ein.

    Annahmen für das Sheet 'Daten':
      - Header- und Metazeilen in den ersten 5 Zeilen (skiprows=5).
      - Spalte B = Schuljahr, Spalte C = Anzahl (usecols=[1, 2]).
      - Leerzeilen am Tabellenende werden via dropna() entfernt.
    """
    df = (pd.read_excel(path, sheet_name='Daten', header=None, usecols=[1, 2], skiprows=5)
            .dropna())
    df.columns = ['Schuljahr', 'Anzahl']
    return df.astype({'Anzahl': int}).reset_index(drop=True)

# Beide Datensätze laden und Zielspalten benennen
lehrer = load(F_LEHRER).rename(columns={'Anzahl': 'Lehrkraefte'})
schueler = load(F_SCHUELER).rename(columns={'Anzahl': 'Schueler'})

# Inner-Join über Schuljahr -> nur Jahre mit beiden Werten bleiben übrig
df = schueler.merge(lehrer, on='Schuljahr', how='inner').sort_values('Schuljahr').reset_index(drop=True)
df

In [ ]:
# Kennzahl: durchschnittliche Schülerzahl je Lehrkraft (rein rechnerisch, ohne Teilzeit-Korrektur)
df['SchuelerProLehrer'] = df['Schueler'] / df['Lehrkraefte']

# Drei-Achsen-Plot: absolute Schüler- und Lehrerzahlen + Verhältnis auf eigener Skala
fig, ax1 = plt.subplots(figsize=(13, 6.5))

color_s = '#1f77b4'  # Schüler   – blau
color_l = '#d62728'  # Lehrer    – rot
color_r = '#2ca02c'  # Verhältnis – grün

# Primärachse: Schülerzahl in Mio.
ax1.plot(df['Schuljahr'], df['Schueler'] / 1e6, color=color_s, linewidth=2.2, marker='o', markersize=4, label='Schüler (Mio.)')
ax1.set_xlabel('Schuljahr')
ax1.set_ylabel('Schüler in Mio.', color=color_s)
ax1.tick_params(axis='y', labelcolor=color_s)
ax1.tick_params(axis='x', rotation=60)
ax1.grid(alpha=0.3)

# Sekundärachse rechts: Lehrkräfte in Tausend
ax2 = ax1.twinx()
ax2.plot(df['Schuljahr'], df['Lehrkraefte'] / 1000, color=color_l, linewidth=2.2, marker='s', markersize=4, label='Lehrkräfte (Tsd.)')
ax2.set_ylabel('Lehrkräfte in Tsd.', color=color_l)
ax2.tick_params(axis='y', labelcolor=color_l)

# Dritte Achse rechts (nach außen versetzt): Verhältnis Schüler/Lehrkraft
ax3 = ax1.twinx()
ax3.spines['right'].set_position(('outward', 60))
ax3.plot(df['Schuljahr'], df['SchuelerProLehrer'], color=color_r, linewidth=2.2, marker='^', markersize=4, linestyle='--', label='Schüler je Lehrkraft')
ax3.set_ylabel('Schüler je Lehrkraft', color=color_r)
ax3.tick_params(axis='y', labelcolor=color_r)

# Gemeinsame Legende für alle drei Achsen
lines = ax1.get_lines() + ax2.get_lines() + ax3.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc='lower left')

plt.title('Schüler, Lehrkräfte & Verhältnis an allgemeinbildenden Schulen', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Verhältnis Schüler/Lehrkraft pro Schuljahr als Tabelle (eine Nachkommastelle)
ratio = df.dropna().assign(SchuelerProLehrer=lambda d: d['Schueler'] / d['Lehrkraefte'])
print(ratio[['Schuljahr', 'SchuelerProLehrer']].to_string(index=False, float_format=lambda x: f'{x:.1f}'))